In [ ]:
!python -m pip install open3d

In [ ]:
import numpy as np
from pathlib import Path
import trimesh

def load_mesh(obj_path: Path) -> list[trimesh.Trimesh]:
    """Load an OBJ and return a list of geometry meshes with visuals/materials.

    Use process=True so trimesh parses the associated MTL and texture maps.
    """
    loaded = trimesh.load(obj_path, process=True)
    if isinstance(loaded, trimesh.Scene):
        # Collect all geometry as trimesh.Trimesh objects
        geoms = []
        for name, geom in loaded.geometry.items():
            if isinstance(geom, trimesh.Trimesh):
                geoms.append(geom)
        return geoms
    elif isinstance(loaded, trimesh.Trimesh):
        return [loaded]
    else:
        return []

# --- paths ---
obj_path = "data/THuman_2.0/0001/0001.obj"
smplx_path = "data/THuman_2.0_smplx_paras/0001/mesh_smplx.obj"

# --- load meshes ---
obj_mesh = load_mesh(obj_path)
smplx_mesh = load_mesh(smplx_path)

In [ ]:
print("Loaded meshes:")
print(f"  OBJ mesh: {obj_mesh[0].vertices.shape[0]} vertices")
print(f"  SMPLX mesh: {smplx_mesh[0].vertices.shape[0]} vertices")

In [ ]:
import torch
import torchvision 

device = "cpu"
print(f"Using device: {device}")

model = torch.jit.load("models/nlf_checkpoint/nlf_l_multi.torchscript", map_location=device).eval()

image_path = "processed/0001/0001_front.png"
image = torchvision.io.read_image(image_path).to(device)
frame_batch = image.unsqueeze(0) # [BatchSize, 3, H, W]
print(frame_batch.shape)

In [ ]:
with torch.inference_mode():
   pred = model.detect_smpl_batched(frame_batch, model_name='smplx')

vertices3d = pred['vertices3d'][0][0]

In [ ]:
import numpy as np
import open3d as o3d

def to_o3d_pcd(points: np.ndarray) -> o3d.geometry.PointCloud:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    return pcd

def diag(points):
    return np.linalg.norm(points.max(0) - points.min(0))

def preprocess(pcd, voxel):
    p = pcd.voxel_down_sample(voxel)
    p.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*2, max_nn=30))
    f = o3d.pipelines.registration.compute_fpfh_feature(
        p, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*5, max_nn=100)
    )
    return p, f

# --- data ---
obj_points = np.asarray(obj_mesh[0].vertices, dtype=np.float64)
# vertices3d_np = vertices3d.cpu().numpy().astype(np.float64) / 1000.0
vertices3d_np = np.asarray(smplx_mesh[0].vertices, dtype=np.float64)

# --- center (don’t mutate originals) ---
obj_c = obj_points - obj_points.mean(axis=0)
tgt_c = vertices3d_np - vertices3d_np.mean(axis=0)

# --- optional: scale match (strongly recommended if diags differ) ---
d_obj = diag(obj_c)
d_tgt = diag(tgt_c)
print("diag obj:", d_obj, "diag tgt:", d_tgt, "ratio obj/tgt:", d_obj/(d_tgt+1e-12))

scale = d_obj / (d_tgt + 1e-12)
tgt_cs = tgt_c * scale

# --- build pcds ---
source = to_o3d_pcd(obj_c)
target = to_o3d_pcd(tgt_cs)

# --- choose voxel/threshold relative to size ---
d = diag(obj_c)
voxel_size = 0.01 * d
threshold  = 0.02 * d

# --- RANSAC init ---
voxel_fpfh = 0.03 * d       # larger than ICP voxel
dist_ransac = 0.06 * d

src_down, src_f = preprocess(source, voxel_fpfh)
tgt_down, tgt_f = preprocess(target, voxel_fpfh)

ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    src_down, tgt_down, src_f, tgt_f,
    mutual_filter=False,
    max_correspondence_distance=dist_ransac,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(dist_ransac),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(500000, 0.999)
)
init_T = ransac.transformation

print("RANSAC fitness-ish:", ransac.fitness if hasattr(ransac, "fitness") else "n/a")
print("RANSAC init:\n", init_T)

# --- ICP refine (p2p then p2l) ---
source_ds = source.voxel_down_sample(voxel_size)
target_ds = target.voxel_down_sample(voxel_size)
source_ds.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2.5, max_nn=30))
target_ds.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2.5, max_nn=30))

res_p2p = o3d.pipelines.registration.registration_icp(
    source_ds, target_ds, threshold, init_T,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=500)
)
res_p2l = o3d.pipelines.registration.registration_icp(
    source_ds, target_ds, threshold, res_p2p.transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

T = res_p2l.transformation
print("ICP fitness:", res_p2l.fitness)
print("ICP RMSE:", res_p2l.inlier_rmse)
print("T:\n", T)

In [ ]:
# verify_icp_camera.py
#
# Purpose:
#   Render point cloud 1 from a chosen camera pose (defined in PC1 coordinates),
#   then transform that SAME camera pose into PC2 coordinates using the ICP
#   transform matrix, and render point cloud 2 from that transformed camera pose.
#
# Output:
#   out/pc1_view.png
#   out/pc2_view.png
#
# Notes:
#   - This uses Open3D's classic Visualizer to render and capture frames.
#   - On macOS you generally want PYOPENGL_PLATFORM=cocoa (NOT egl/osmesa).
#
# Usage examples:
#   python verify_icp_camera.py --pc1 data/pc1.ply --pc2 data/pc2.ply --T data/T_1_to_2.npy
#   python verify_icp_camera.py --pc1 data/pc1.npy --pc2 data/pc2.npy --T data/T.txt --outdir out
#
# T convention:
#   T_1_to_2 maps points from cloud1 coords to cloud2 coords:
#     p2 = T_1_to_2 @ [p1, 1]

import argparse
import os
from pathlib import Path
import numpy as np
import open3d as o3d


def load_points(path: str) -> np.ndarray:
    """
    Loads Nx3 points from:
      - .npy (Nx3 float array)
      - .ply/.pcd/.xyz/.xyzn/.xyzrgb via Open3D
    """
    p = Path(path)
    suf = p.suffix.lower()
    if suf == ".npy":
        pts = np.load(str(p))
        pts = np.asarray(pts, dtype=np.float64)
        if pts.ndim != 2 or pts.shape[1] != 3:
            raise ValueError(f"{path}: expected Nx3 array, got shape {pts.shape}")
        return pts

    pcd = o3d.io.read_point_cloud(str(p))
    if pcd.is_empty():
        raise ValueError(f"{path}: Open3D read empty point cloud.")
    return np.asarray(pcd.points, dtype=np.float64)


def points_to_pcd(pts: np.ndarray) -> o3d.geometry.PointCloud:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd


def load_T(path: str) -> np.ndarray:
    """
    Loads 4x4 transform from:
      - .npy (4x4)
      - .txt / .csv whitespace or comma separated 4x4
    """
    p = Path(path)
    suf = p.suffix.lower()
    if suf == ".npy":
        T = np.load(str(p)).astype(np.float64)
    else:
        txt = Path(path).read_text().strip().replace(",", " ")
        vals = [float(x) for x in txt.split()]
        if len(vals) != 16:
            raise ValueError(f"{path}: expected 16 numbers for 4x4, got {len(vals)}")
        T = np.array(vals, dtype=np.float64).reshape(4, 4)

    if T.shape != (4, 4):
        raise ValueError(f"{path}: expected 4x4, got {T.shape}")
    return T


def transform_point(T: np.ndarray, p: np.ndarray) -> np.ndarray:
    ph = np.array([p[0], p[1], p[2], 1.0], dtype=np.float64)
    qh = T @ ph
    return qh[:3] / (qh[3] if qh[3] != 0 else 1.0)


def transform_direction(T: np.ndarray, v: np.ndarray) -> np.ndarray:
    # directions don't translate, only rotate (and scale if present; ICP rigid should be pure rotation)
    R = T[:3, :3]
    return R @ v


def compute_default_camera_from_bounds(pts: np.ndarray):
    """
    Returns (eye, lookat, up) in the point cloud's coordinate system.
    This is a decent default if you don't want to hardcode a camera.
    """
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    center = (mins + maxs) * 0.5
    extent = np.linalg.norm(maxs - mins)
    if extent == 0:
        extent = 1.0

    # Put the camera "in front" along +Z, looking towards the center.
    eye = center + np.array([0.0, 0.0, 2.0 * extent])
    lookat = center
    up = np.array([0.0, 1.0, 0.0])
    return eye, lookat, up


def render_from_camera(
    geometry: o3d.geometry.Geometry,
    eye: np.ndarray,
    lookat: np.ndarray,
    up: np.ndarray,
    out_path: str,
    width: int = 1280,
    height: int = 720,
    point_size: float = 2.0,
    show_window: bool = False,
):
    """
    Renders `geometry` with a camera defined by (eye, lookat, up) using Open3D Visualizer
    and saves a PNG.
    """
    vis = o3d.visualization.Visualizer()
    vis.create_window(
        window_name="verify_icp_camera",
        width=width,
        height=height,
        visible=show_window,
    )
    vis.add_geometry(geometry)

    opt = vis.get_render_option()
    opt.point_size = float(point_size)
    opt.background_color = np.asarray([0, 0, 0], dtype=np.float64)

    ctr = vis.get_view_control()
    ctr.set_lookat(lookat.tolist())
    ctr.set_front((lookat - eye).tolist())  # camera front points from eye -> lookat
    ctr.set_up(up.tolist())

    # Make sure frames are updated before capture
    vis.poll_events()
    vis.update_renderer()

    # Capture
    img = vis.capture_screen_float_buffer(do_render=True)
    img = (np.asarray(img) * 255.0).clip(0, 255).astype(np.uint8)

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    o3d.io.write_image(out_path, o3d.geometry.Image(img))

    vis.destroy_window()



def verify_icp_camera_views(
    source: o3d.geometry.PointCloud,
    target: o3d.geometry.PointCloud,
    T: np.ndarray,
    *,
    outdir: str = "out",
    width: int = 1280,
    height: int = 720,
    point_size: float = 2.0,
    show: bool = False,
    pts1: np.ndarray | None = None,
):
    """
    Render source from a camera pose defined in source coordinates, then transform that
    camera pose with T (source->target) and render target from the transformed pose.

    Parameters
    ----------
    source, target : open3d.geometry.PointCloud
    T : (4,4) ndarray
        Transform mapping source coords -> target coords. p2 = T @ [p1,1]
    outdir : str
        Output directory for screenshots.
    width, height : int
        Render resolution.
    point_size : float
        Point size in pixels.
    show : bool
        If True, show windows while rendering (usually False in notebooks).
    pts1 : (N,3) ndarray | None
        Optional raw points for computing bounds/camera. If None, uses source.points.
    """
    if T.shape != (4, 4):
        raise ValueError(f"T must be 4x4, got {T.shape}")

    # Points for camera auto-placement
    if pts1 is None:
        pts1 = np.asarray(source.points, dtype=np.float64)
    if pts1.size == 0:
        raise ValueError("source point cloud is empty")

    # Define a camera in PC1 coordinates
    eye1, lookat1, up1 = compute_default_camera_from_bounds(pts1)

    # Transform camera into PC2 coordinates
    eye2 = transform_point(T, eye1)
    lookat2 = transform_point(T, lookat1)
    up2 = transform_direction(T, up1)
    up2 = up2 / (np.linalg.norm(up2) + 1e-12)

    outdir_p = Path(outdir)
    outdir_p.mkdir(parents=True, exist_ok=True)
    out1 = str(outdir_p / "pc1_view.png")
    out2 = str(outdir_p / "pc2_view.png")

    render_from_camera(
        source, eye1, lookat1, up1,
        out_path=out1,
        width=width, height=height,
        point_size=point_size,
        show_window=show,
    )

    render_from_camera(
        target, eye2, lookat2, up2,
        out_path=out2,
        width=width, height=height,
        point_size=point_size,
        show_window=show,
    )

    print("Saved:")
    print(" ", out1)
    print(" ", out2)
    print("\nCamera (PC1):")
    print(" eye   =", eye1)
    print(" look  =", lookat1)
    print(" up    =", up1)
    print("\nCamera transformed to PC2 with T_1_to_2:")
    print(" eye   =", eye2)
    print(" look  =", lookat2)
    print(" up    =", up2)

    return {
        "pc1_image": out1,
        "pc2_image": out2,
        "eye1": eye1, "lookat1": lookat1, "up1": up1,
        "eye2": eye2, "lookat2": lookat2, "up2": up2,
    }




In [ ]:
obj_c_o3d = to_o3d_pcd(obj_c)
tgt_c_o3d = to_o3d_pcd(tgt_cs)

res = verify_icp_camera_views(source=obj_c_o3d, target=tgt_c_o3d, T=T, outdir="out", point_size=2.0, show=False)
res["pc1_image"], res["pc2_image"]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_pc1_with_pc2_transformed(
    pc1: np.ndarray,
    pc2: np.ndarray,
    T_1_to_2: np.ndarray,
    *,
    T_maps: str = "pc1_to_pc2",   # "pc1_to_pc2" or "pc2_to_pc1"
    max_points: int = 5_000,    # downsample for speed
    s: float = 0.5,               # marker size
    elev: float = 15,
    azim: float = -60,
):
    """
    Plots pc1 and pc2 transformed into pc1 coordinate system.

    Convention:
      If T_maps="pc1_to_pc2", then pc2_in_pc1 = inv(T_1_to_2) applied to pc2.
      If T_maps="pc2_to_pc1", then pc2_in_pc1 = T_1_to_2 applied to pc2.
    """
    pc1 = np.asarray(pc1, dtype=np.float64)
    pc2 = np.asarray(pc2, dtype=np.float64)
    T = np.asarray(T_1_to_2, dtype=np.float64)

    if pc1.ndim != 2 or pc1.shape[1] != 3:
        raise ValueError(f"pc1 must be Nx3, got {pc1.shape}")
    if pc2.ndim != 2 or pc2.shape[1] != 3:
        raise ValueError(f"pc2 must be Nx3, got {pc2.shape}")
    if T.shape != (4, 4):
        raise ValueError(f"T must be 4x4, got {T.shape}")

    def apply_T(points: np.ndarray, T44: np.ndarray) -> np.ndarray:
        ones = np.ones((points.shape[0], 1), dtype=np.float64)
        ph = np.hstack([points, ones])                  # (N,4)
        qh = (T44 @ ph.T).T                             # (N,4)
        return qh[:, :3] / np.where(qh[:, 3:4] == 0, 1.0, qh[:, 3:4])

    # transform pc2 -> pc1 coords
    if T_maps == "pc1_to_pc2":
        T_pc2_to_pc1 = np.linalg.inv(T)
    elif T_maps == "pc2_to_pc1":
        T_pc2_to_pc1 = T
    else:
        raise ValueError("T_maps must be 'pc1_to_pc2' or 'pc2_to_pc1'")

    pc2_in_pc1 = apply_T(pc2, T_pc2_to_pc1)

    # downsample for plotting
    def maybe_sample(pts: np.ndarray) -> np.ndarray:
        if pts.shape[0] <= max_points:
            return pts
        idx = np.random.choice(pts.shape[0], size=max_points, replace=False)
        return pts[idx]

    pc1p = maybe_sample(pc1)
    pc2p = maybe_sample(pc2_in_pc1)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(pc1p[:, 0], pc1p[:, 1], pc1p[:, 2], s=s, label="pc1 (original)")
    ax.scatter(pc2p[:, 0], pc2p[:, 1], pc2p[:, 2], s=s, label="pc2 -> pc1 coords")

    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend(loc="upper right")

    # equal aspect (matplotlib 3.3+)
    all_pts = np.vstack([pc1p, pc2p])
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    centers = (mins + maxs) / 2
    ranges = (maxs - mins)
    r = float(np.max(ranges) / 2) if np.max(ranges) > 0 else 1.0
    ax.set_xlim(centers[0] - r, centers[0] + r)
    ax.set_ylim(centers[1] - r, centers[1] + r)
    ax.set_zlim(centers[2] - r, centers[2] + r)

    plt.tight_layout()
    plt.show()

    return pc2_in_pc1

pc2_in_pc1 = plot_pc1_with_pc2_transformed(obj_c, tgt_cs, T, T_maps="pc1_to_pc2")
